# Germany monetary policy: source coverage audit

This notebook verifies the retained public source snapshot and checks whether the production, inflation and shock series overlap. No policy responses are estimated and no synthetic observations are generated.

The success criteria are unchanged source hashes, valid date keys and an explicit account of available observations and gaps. A long sample alone does not establish identification or statistical power.

Sources: [project source register](../SOURCES.md) and [data audit](../data_audit.md). The snapshot was retrieved on 15 September 2026. Eurostat supplies official indices; the Jarociński update supplies research estimates of ECB monetary-policy and information shocks.

In [1]:
from pathlib import Path
import json

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
import runpy
from IPython.display import Markdown, display

audit_module = runpy.run_path(str(ROOT / "scripts/audit_germany_data.py"))
report = audit_module["audit"]()
print("Source files passing checksum verification:", report["verified_source_files"])
print(report["interpretation"])


Source files passing checksum verification: 5
Coverage only; no causal estimates, imputation or statistical-power claim.


## Coverage and common sample

The table counts non-missing observations in each source. The final row intersects the three index series and the monthly shocks. Lag construction, transformations, controls and response horizons can subsequently shorten this interval.

In [2]:
table = ["| Series | Observations | First | Last | Internal gaps |",
         "| --- | ---: | --- | --- | ---: |"]
for name, summary in report.items():
    if isinstance(summary, dict) and "observations" in summary:
        gaps = len(summary["missing_within_range"]) if "missing_within_range" in summary else "Event frequency"
        table.append(f"| {name.replace('_', ' ')} | {summary['observations']} | {summary['first']} | {summary['last']} | {gaps} |")
display(Markdown("\n".join(table)))


| Series | Observations | First | Last | Internal gaps |
| --- | ---: | --- | --- | ---: |
| durable production | 427 | 1991-01 | 2026-07 | 0 |
| nondurable production | 427 | 1991-01 | 2026-07 | 0 |
| hicp | 360 | 1996-01 | 2025-12 | 0 |
| monthly shocks | 322 | 1999-01 | 2025-10 | 0 |
| event shocks | 312 | 1999-01-07 | 2025-10-30 | Event frequency |
| common months before transformations | 322 | 1999-01 | 2025-10 | 0 |

## What the indices measure

`MIG_DCOG` and `MIG_NDCOG` are durable and nondurable consumer-goods production groups. They are seasonally and calendar adjusted volume indices with 2021=100. They do not measure household consumption and do not reproduce the broader US manufacturing classification.

HICP is the historical German all-items index with 2015=100. Monthly shocks retain the provider's definitions and normalisation. Recorded zeros must remain distinct from missing observations and excluded events.

## Remaining work

Reconcile the raw EA-MPD event coverage with the derived shock file, review identification assumptions and declare the primary estimand, sample rules and inference procedure before fitting the main model. The source snapshot is a current retrieval of historical values, not a reconstruction of earlier publication vintages.

The audit confirms accessible data and a 322-month overlap before transformations. It supports proceeding to the research-design stage. It is not an empirical result about the effect of ECB tightening.